<a href="https://colab.research.google.com/github/alexoveras/dashboard-fraudes/blob/main/ProjetoFraudes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas openpyxl

In [2]:
import pandas as pd

df = pd.read_csv("creditcard.csv")

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
import pandas as pd
import numpy as np

# ==============================
# DIMENSÃO TEMPO
# ==============================

dim_tempo = pd.DataFrame()

dim_tempo["TIME"] = sorted(df["Time"].unique())

dim_tempo["ID_TEMPO"] = range(
    1,
    len(dim_tempo) + 1
)

dim_tempo["HORA"] = (
    dim_tempo["TIME"] // 3600
).astype(int) % 24


def periodo_dia(hora):

    if hora < 6:
        return "Madrugada"

    elif hora < 12:
        return "Manhã"

    elif hora < 18:
        return "Tarde"

    return "Noite"


dim_tempo["PERIODO_DIA"] = (
    dim_tempo["HORA"]
    .apply(periodo_dia)
)

print(dim_tempo.head())
print("\nTotal registros:")
print(len(dim_tempo))

   TIME  ID_TEMPO  HORA PERIODO_DIA
0   0.0         1     0   Madrugada
1   1.0         2     0   Madrugada
2   2.0         3     0   Madrugada
3   4.0         4     0   Madrugada
4   7.0         5     0   Madrugada

Total registros:
124592


In [4]:
dim_tempo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124592 entries, 0 to 124591
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   TIME         124592 non-null  float64
 1   ID_TEMPO     124592 non-null  int64  
 2   HORA         124592 non-null  int64  
 3   PERIODO_DIA  124592 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 3.8+ MB


In [5]:
# ==============================
# DIMENSÃO FAIXA DE VALOR
# ==============================

faixas = [
    (1, "Até R$10", 0, 10),
    (2, "R$10 a R$50", 10, 50),
    (3, "R$50 a R$100", 50, 100),
    (4, "R$100 a R$250", 100, 250),
    (5, "R$250 a R$500", 250, 500),
    (6, "R$500 a R$1000", 500, 1000),
    (7, "Acima de R$1000", 1000, 999999)
]

dim_faixa_valor = pd.DataFrame(
    faixas,
    columns=[
        "ID_VALOR",
        "FAIXA_VALOR",
        "VALOR_MIN",
        "VALOR_MAX"
    ]
)

dim_faixa_valor

,ID_VALOR,FAIXA_VALOR,VALOR_MIN,VALOR_MAX
0,1,Até R$10,0,10
1,2,R$10 a R$50,10,50
2,3,R$50 a R$100,50,100
3,4,R$100 a R$250,100,250
4,5,R$250 a R$500,250,500
5,6,R$500 a R$1000,500,1000
6,7,Acima de R$1000,1000,999999


In [6]:
# ==============================
# DIMENSÃO RISCO
# ==============================

dim_risco = pd.DataFrame({
    "ID_RISCO": [1, 2],
    "CLASSIFICACAO": [
        "Transação Normal",
        "Fraude"
    ],
    "CLASS": [0, 1]
})

dim_risco

,ID_RISCO,CLASSIFICACAO,CLASS
0,1,Transação Normal,0
1,2,Fraude,1


In [7]:
# =================================
# CRIANDO A FATO_TRANSACAO
# =================================

fato = df.copy()

fato.insert(
    0,
    "ID_TRANSACAO",
    range(1, len(fato)+1)
)

# Relacionamento com DIM_TEMPO

mapa_tempo = dict(
    zip(
        dim_tempo["TIME"],
        dim_tempo["ID_TEMPO"]
    )
)

fato["ID_TEMPO"] = (
    fato["Time"]
    .map(mapa_tempo)
)

# Classificação Faixa Valor

def faixa_valor(valor):

    if valor <= 10:
        return 1

    elif valor <= 50:
        return 2

    elif valor <= 100:
        return 3

    elif valor <= 250:
        return 4

    elif valor <= 500:
        return 5

    elif valor <= 1000:
        return 6

    return 7

fato["ID_VALOR"] = (
    fato["Amount"]
    .apply(faixa_valor)
)

# Classificação Risco

fato["ID_RISCO"] = (
    fato["Class"]
    .apply(lambda x: 2 if x == 1 else 1)
)

print(fato.shape)

fato.head()

(284807, 35)


,ID_TRANSACAO,Time,V1,V2,V3,V4,V5,V6,V7,V8,...,V24,V25,V26,V27,V28,Amount,Class,ID_TEMPO,ID_VALOR,ID_RISCO
0,1,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,...,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0,1,4,1
1,2,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,...,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0,1,1,1
2,3,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,...,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,2,5,1
3,4,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,...,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0,2,4,1
4,5,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,...,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0,3,3,1


In [8]:
fato.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 35 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   ID_TRANSACAO  284807 non-null  int64  
 1   Time          284807 non-null  float64
 2   V1            284807 non-null  float64
 3   V2            284807 non-null  float64
 4   V3            284807 non-null  float64
 5   V4            284807 non-null  float64
 6   V5            284807 non-null  float64
 7   V6            284807 non-null  float64
 8   V7            284807 non-null  float64
 9   V8            284807 non-null  float64
 10  V9            284807 non-null  float64
 11  V10           284807 non-null  float64
 12  V11           284807 non-null  float64
 13  V12           284807 non-null  float64
 14  V13           284807 non-null  float64
 15  V14           284807 non-null  float64
 16  V15           284807 non-null  float64
 17  V16           284807 non-null  float64
 18  V17 

In [9]:
print("FATO")
print(fato.shape)

print("\nDIM_TEMPO")
print(dim_tempo.shape)

print("\nDIM_FAIXA_VALOR")
print(dim_faixa_valor.shape)

print("\nDIM_RISCO")
print(dim_risco.shape)

FATO
(284807, 35)

DIM_TEMPO
(124592, 4)

DIM_FAIXA_VALOR
(7, 4)

DIM_RISCO
(2, 3)


In [10]:
# ==========================
# EXPORTAÇÃO CSV
# ==========================

fato.to_csv(
    "transacoes.csv",
    index=False
)

dim_tempo.to_csv(
    "dim_tempo.csv",
    index=False
)

dim_faixa_valor.to_csv(
    "dim_faixa_valor.csv",
    index=False
)

dim_risco.to_csv(
    "dim_risco.csv",
    index=False
)

# ==========================
# EXPORTAÇÃO EXCEL
# ==========================

with pd.ExcelWriter(
    "base_bi_fraudes.xlsx",
    engine="openpyxl"
) as writer:

    fato.to_excel(
        writer,
        sheet_name="TRANSACOES",
        index=False
    )

    dim_tempo.to_excel(
        writer,
        sheet_name="DIM_TEMPO",
        index=False
    )

    dim_faixa_valor.to_excel(
        writer,
        sheet_name="DIM_FAIXA_VALOR",
        index=False
    )

    dim_risco.to_excel(
        writer,
        sheet_name="DIM_RISCO",
        index=False
    )

print("✅ Arquivos gerados com sucesso!")


✅ Arquivos gerados com sucesso!
